# Multi-Agent Patterns with LangGraph + Ollama

Two **real-time** multi-agent scenarios, each showing a *different* collaboration pattern (all local Ollama models, free key-less public APIs):

| | Scenario A: Travel Desk | Scenario B: Global Situation Room |
|---|---|---|
| Pattern | **Supervisor / router** — an LLM decides who works next | **Parallel fan-out / fan-in** — specialists work at the same time, an editor merges |
| Agents | Weather, Currency, City-Guide specialists + Writer | Quake Analyst, Weather Analyst, Markets Analyst + Editor |
| Live data | Open-Meteo, Frankfurter (FX), Wikipedia | USGS earthquakes, Open-Meteo, Frankfurter (FX) |
| LangGraph concepts | conditional edges, structured output, loops back to supervisor | multiple edges from `START`, join node, reducer-free parallel state keys |

(The first notebook in this folder covers a single agent and a sequential writer/editor team; this one goes deeper into multi-agent *topologies*.)

## 0. Setup

In [ ]:
%pip install -q langgraph langchain-ollama langchain-core pydantic requests

Pull a tool/structured-output capable model first:

```bash
ollama pull llama3.2
```

In [ ]:
import time, datetime, operator, requests
from typing import Annotated, Literal, Optional, TypedDict

from pydantic import BaseModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

MODEL = "llama3.2"      # any local model that supports structured output / tools
llm = ChatOllama(model=MODEL, temperature=0)
HEADERS = {"User-Agent": "agent-notebook-demo/1.0"}

def chat(system, user):
    return llm.invoke([SystemMessage(system), HumanMessage(user)]).content

print(chat("Reply in one word.", "Are you ready?"))

---
# Scenario A — Supervisor Pattern: the Travel Desk

**Situation:** a traveller types a free-text request. A **supervisor** LLM reads it and decides *which specialist to call next* — and when to stop. Only relevant specialists run, so
*"Just the weather in Oslo"* skips currency and city-guide, while *"Flying to Tokyo from India"* uses all three.

```
              ┌──────────────▶ weather_agent ─┐
 START → parse → supervisor ──▶ currency_agent ─┼─(back to supervisor)
              ▲  └────────────▶ guide_agent ───┘
              │                      │
              └──────────────────────┘ when done ──▶ writer → END
```

### 🤖 Agent profiles — the Travel Desk
**Request parser** (`parse_node`)
- **Goal:** turn a free-text trip request into clean fields (city, destination country, home country).
- **Input:** the traveller's sentence. **Tools:** none; it uses the LLM's structured output (a Pydantic schema).
- **Output:** city plus currency codes, which our own country-to-currency table fills in because small models get ISO codes wrong.

**Supervisor** (`supervisor_node`)
- **Goal:** decide *which specialists are actually needed* and dispatch them one by one, then hand over to the Writer.
- **Input:** the request and which specialists have already reported. **Tools:** none; it returns structured yes/no flags (`needs_weather`, `needs_currency`, `needs_guide`).
- **Output:** the name of the next node to run. **Why it exists:** irrelevant specialists are skipped, which saves time and keeps the answer focused.

**Weather Specialist** (`weather_agent`)
- **Goal:** report the current weather and what to pack. **Input:** the city. **Tools:** Open-Meteo weather API.
- **Output:** 2 short lines (temperature range, rain chance, packing advice).

**Currency Specialist** (`currency_agent`)
- **Goal:** tell the traveller how their money converts. **Input:** home and destination currency. **Tools:** Frankfurter exchange-rate API.
- **Output:** 2 short lines with the live rate and a budgeting tip.

**City Guide** (`guide_agent`)
- **Goal:** say what the city is known for and one thing to do. **Input:** the city name. **Tools:** Wikipedia summary API.
- **Output:** 2-3 short lines grounded in the Wikipedia extract.

**Writer** (`writer_node`)
- **Goal:** merge the specialists' notes into one friendly briefing. **Input:** only the notes, so it must not add facts from memory.
- **Tools:** none. **Output:** a briefing of at most 6 lines that mentions only the topics the specialists covered.

### Tools (real-time data)

In [ ]:
def geocode(city):
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}, timeout=10).json()
    return r["results"][0] if r.get("results") else None

def tool_weather(city: str) -> str:
    p = geocode(city)
    if not p: return f"Could not find '{city}'."
    d = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": p["latitude"], "longitude": p["longitude"], "timezone": "auto",
        "current": "temperature_2m,wind_speed_10m",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}, timeout=10).json()
    return (f"{p['name']}: now {d['current']['temperature_2m']}C, wind {d['current']['wind_speed_10m']} km/h; "
            f"today {d['daily']['temperature_2m_min'][0]}-{d['daily']['temperature_2m_max'][0]}C, "
            f"rain chance {d['daily']['precipitation_probability_max'][0]}%")

def tool_fx(base: str, target: str) -> str:
    r = requests.get("https://api.frankfurter.dev/v1/latest", params={"base": base, "symbols": target}, timeout=10).json()
    if "rates" not in r: return f"No rate for {base}->{target}."
    return f"1 {base} = {r['rates'][target]} {target} (as of {r['date']})"

def tool_wiki(topic: str) -> str:
    r = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic.replace(' ', '_')}", headers=HEADERS, timeout=10)
    return r.json().get("extract", "No summary found.")[:600] if r.ok else "No summary found."

print(tool_weather("Tokyo")); print(tool_fx("INR", "JPY")); print(tool_wiki("Tokyo")[:120], "...")

### State, request parsing and the supervisor

`parse_node` uses **structured output** (a Pydantic schema) to pull the city and countries out of free text; we map countries to currency codes ourselves because small models are unreliable at ISO codes.
The supervisor also returns structured output — a set of yes/no flags saying which specialists the request needs — and then dispatches them one by one, coming back to the supervisor after each.

In [ ]:
WORKERS = ["weather_agent", "currency_agent", "guide_agent"]

# Small models are unreliable at ISO currency codes, so we ask for COUNTRIES and map them ourselves.
CURRENCY = {"india": "INR", "japan": "JPY", "france": "EUR", "germany": "EUR", "italy": "EUR", "spain": "EUR",
            "united states": "USD", "usa": "USD", "us": "USD", "united kingdom": "GBP", "uk": "GBP",
            "australia": "AUD", "canada": "CAD", "switzerland": "CHF", "singapore": "SGD", "thailand": "THB",
            "norway": "NOK", "china": "CNY", "brazil": "BRL", "south korea": "KRW"}

class TripRequest(BaseModel):
    city: str
    destination_country: Optional[str] = None
    home_country: Optional[str] = None       # where the traveller lives / is flying from

class Needs(BaseModel):
    needs_weather: bool
    needs_currency: bool
    needs_guide: bool

class DeskState(TypedDict):
    request: str
    trip: dict
    needed: list           # specialists the supervisor decided are relevant
    reports: dict          # worker name -> its findings
    next: str
    final: str
    trace: list            # order in which agents ran

def parse_node(state: DeskState):
    t = llm.with_structured_output(TripRequest).invoke(
        f"Extract the destination city, its country, and the traveller's home country (if stated) from: {state['request']}")
    trip = t.model_dump()
    trip["home_currency"] = CURRENCY.get((t.home_country or "").lower())
    trip["dest_currency"] = CURRENCY.get((t.destination_country or "").lower())
    print("parsed ->", trip)
    return {"trip": trip, "reports": {}, "trace": [], "needed": []}

SUPERVISOR = (
    "You are a travel-desk supervisor. Decide which specialists are needed to answer the traveller's request. "
    "needs_weather: they ask about weather/forecast/packing. needs_currency: they ask about money/exchange rates/budget. "
    "needs_guide: they ask what the city is known for / sights / things to do. Set a flag true ONLY if the request asks for it."
)

def supervisor_node(state: DeskState):
    needed = state["needed"]
    if not needed and not state["reports"]:                 # first visit: the LLM plans who is needed
        n = llm.with_structured_output(Needs).invoke([SystemMessage(SUPERVISOR), HumanMessage(state["request"])])
        needed = [w for w, flag in zip(WORKERS, [n.needs_weather, n.needs_currency, n.needs_guide]) if flag]
        print(f"[supervisor] plan -> {needed}")
    todo = [w for w in needed if w not in state["reports"]]
    choice = todo[0] if todo else "writer"                  # dispatch next pending specialist, else finish
    print(f"[supervisor] next -> {choice}")
    return {"needed": needed, "next": choice}

def route_from_supervisor(state: DeskState):
    return state["next"]

### Specialist agents and the writer

In [ ]:
def make_worker(key, persona, fetch):
    def node(state: DeskState):
        data = fetch(state["trip"])
        note = chat(persona, f"Request: {state['request']}\nData: {data}")
        print(f"[{key}] data: {data[:110]}")
        return {"reports": {**state["reports"], key: note}, "trace": state["trace"] + [key]}
    node.__name__ = key
    return node

def fetch_weather(t): return tool_weather(t["city"])
def fetch_fx(t):
    if not (t.get("home_currency") and t.get("dest_currency")): return "Currencies not specified."
    return tool_fx(t["home_currency"].upper(), t["dest_currency"].upper())
def fetch_guide(t): return tool_wiki(t["city"])

weather_agent  = make_worker("weather_agent",  "You are a Weather Specialist. In 2 short lines, summarise the weather data and what to pack.", fetch_weather)
currency_agent = make_worker("currency_agent", "You are a Currency Specialist. In 2 short lines, state the exchange rate and a quick budgeting tip.", fetch_fx)
guide_agent    = make_worker("guide_agent",    "You are a City Guide. In 2-3 short lines, say what the city is known for and one thing to do.", fetch_guide)

def writer_node(state: DeskState):
    notes = "\n".join(f"- {k}: {v}" for k, v in state["reports"].items())
    final = chat("You are a Travel Desk Writer. Combine the specialists' notes into one short, friendly briefing (max 6 lines). "
                 "Use ONLY facts that appear in the notes - never add facts from memory, and never mention topics the notes don't cover.",
                 f"Request: {state['request']}\nNotes:\n{notes}")
    return {"final": final}

### Wire the graph

In [ ]:
g = StateGraph(DeskState)
g.add_node("parse", parse_node)
g.add_node("supervisor", supervisor_node)
g.add_node("weather_agent", weather_agent)
g.add_node("currency_agent", currency_agent)
g.add_node("guide_agent", guide_agent)
g.add_node("writer", writer_node)

g.add_edge(START, "parse")
g.add_edge("parse", "supervisor")
g.add_conditional_edges("supervisor", route_from_supervisor,
                        {"weather_agent": "weather_agent", "currency_agent": "currency_agent",
                         "guide_agent": "guide_agent", "writer": "writer"})
for w in WORKERS:
    g.add_edge(w, "supervisor")        # every specialist reports back to the supervisor
g.add_edge("writer", END)
travel_desk = g.compile()

travel_desk     # renders the graph in Jupyter

In [ ]:
def desk(request):
    print("REQUEST:", request, "\n")
    out = travel_desk.invoke({"request": request, "trip": {}, "needed": [], "reports": {}, "next": "", "final": "", "trace": []})
    print("\nAgents that ran:", out["trace"])
    print("\n" + out["final"] + "\n" + "=" * 70)

desk("I'm flying from India to Tokyo next week. What's the weather, how much is a yen in rupees, and what is Tokyo known for?")

**Now watch the supervisor skip work.** Same graph, narrower request:

In [ ]:
desk("Just tell me the weather in Oslo right now.")
desk("I live in the US and am visiting Paris - what's the euro rate for me?")

---
# Scenario B — Parallel Fan-out / Fan-in: the Global Situation Room

**Situation:** every morning an editor wants a 5-line "world snapshot". Three analysts gather **independent** live data at the same time; the Editor waits for all three, then merges.

```
            ┌─▶ quake_analyst ───┐
 START ─────┼─▶ weather_analyst ─┼──▶ editor → END
            └─▶ markets_analyst ─┘
```

**Why parallel:** the analysts don't depend on each other, so there's no reason to run them in sequence. In LangGraph you just add several edges out of `START`; the node with several incoming edges (`editor`) runs only when *all* of them are done.
Each analyst writes to its **own** state key, so parallel writes never collide.

### 🤖 Agent profiles — the Situation Room
**Quake Analyst** (`quake_analyst`)
- **Goal:** summarise what the Earth did in the last 24 hours. **Input:** live USGS data for earthquakes of magnitude 4.5 and above.
- **Tools:** USGS earthquake feed. **Output:** 2 lines with the number of quakes and the strongest event.
- **Why separate:** it works on completely independent data, so it can run at the same time as the others.

**Weather Analyst** (`weather_analyst`)
- **Goal:** compare current conditions in four world cities (London, New York, Mumbai, Tokyo). **Tools:** Open-Meteo weather API.
- **Output:** 2 lines naming the warmest and the wettest city.

**Markets Analyst** (`markets_analyst`)
- **Goal:** show which currency moved most against the US dollar this week. **Tools:** Frankfurter API (today versus 7 days ago).
- **Output:** 2 lines with the biggest mover and its percentage change.

**Editor** (`editor`)
- **Goal:** merge the three reports into a "Global Snapshot": one headline plus 3 bullets (Earth, Weather, Markets).
- **Input:** all three analyst reports. It runs only after *every* analyst has finished (fan-in).
- **Tools:** none. **Output:** the final briefing, using only facts from the reports.

In [ ]:
def tool_quakes() -> str:
    r = requests.get("https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_day.geojson", timeout=10).json()
    feats = sorted(r["features"], key=lambda f: -f["properties"]["mag"])[:5]
    lines = [f"M{f['properties']['mag']} - {f['properties']['place']}" for f in feats]
    return f"{r['metadata']['count']} quakes of M4.5+ in the last 24h. Strongest:\n" + "\n".join(lines)

def tool_city_weather_snapshot() -> str:
    return "\n".join(tool_weather(c) for c in ["London", "New York", "Mumbai", "Tokyo"])

def tool_fx_trend() -> str:
    week_ago = (datetime.date.today() - datetime.timedelta(days=7)).isoformat()
    now = requests.get("https://api.frankfurter.dev/v1/latest", params={"base": "USD", "symbols": "EUR,GBP,INR,JPY"}, timeout=10).json()
    old = requests.get(f"https://api.frankfurter.dev/v1/{week_ago}", params={"base": "USD", "symbols": "EUR,GBP,INR,JPY"}, timeout=10).json()
    return "\n".join(f"USD/{c}: {old['rates'][c]} -> {now['rates'][c]} ({(now['rates'][c]/old['rates'][c]-1)*100:+.2f}% in a week)"
                      for c in now["rates"])

print(tool_quakes()); print(tool_fx_trend())

In [ ]:
class RoomState(TypedDict):
    quake_report: str
    weather_report: str
    markets_report: str
    briefing: str
    timings: Annotated[list, operator.add]     # reducer: parallel branches can all append

def analyst(key, persona, fetch):
    def node(state: RoomState):
        t0 = time.time()
        data = fetch()
        text = chat(persona, f"Live data:\n{data}")
        dt = time.time() - t0
        print(f"[{key}] finished in {dt:.1f}s")
        return {key: text, "timings": [(key, round(dt, 1))]}
    node.__name__ = key
    return node

quake_analyst   = analyst("quake_report",   "You are a Seismology Analyst. In 2 lines, summarise the earthquake data; note the strongest event.", tool_quakes)
weather_analyst = analyst("weather_report", "You are a Weather Analyst. In 2 lines, compare these cities: who has the warmest and the wettest conditions?", tool_city_weather_snapshot)
markets_analyst = analyst("markets_report", "You are an FX Markets Analyst. In 2 lines, say which currency moved most against USD this week.", tool_fx_trend)

def editor_node(state: RoomState):
    print("[editor] merging all three reports")
    briefing = chat(
        "You are the Editor of a daily 'Global Snapshot'. Merge the three analyst reports into exactly 3 bullets "
        "(Earth, Weather, Markets) plus a one-line headline. Use only the reports.",
        f"Quakes: {state['quake_report']}\n\nWeather: {state['weather_report']}\n\nMarkets: {state['markets_report']}")
    return {"briefing": briefing}

room = StateGraph(RoomState)
room.add_node("quake_analyst", quake_analyst)
room.add_node("weather_analyst", weather_analyst)
room.add_node("markets_analyst", markets_analyst)
room.add_node("editor", editor_node)
for a in ["quake_analyst", "weather_analyst", "markets_analyst"]:
    room.add_edge(START, a)         # fan-out: all start together
    room.add_edge(a, "editor")      # fan-in: editor waits for all three
room.add_edge("editor", END)
situation_room = room.compile()

situation_room

In [ ]:
t0 = time.time()
result = situation_room.invoke({"quake_report": "", "weather_report": "", "markets_report": "", "briefing": "", "timings": []})
print(f"\nTotal wall time: {time.time() - t0:.1f}s   (sum of analyst times: {sum(t for _, t in result['timings']):.1f}s)\n")

from IPython.display import Markdown
Markdown(result["briefing"])

> **About the timing:** the *data fetching* runs in parallel, but a single local Ollama server often handles one generation at a time by default, so LLM calls may still queue.
> To let them overlap, start Ollama with `OLLAMA_NUM_PARALLEL=3` (needs enough RAM/VRAM). With hosted models the speed-up is much larger.

---
# Choosing a multi-agent topology

| Pattern | Use when | Strength | Watch out for |
|---|---|---|---|
| **Sequential pipeline** (first notebook) | Fixed stages: research → write → review | Simple, predictable | No flexibility |
| **Supervisor / router** (Scenario A) | Which specialists are needed depends on the request | Skips unnecessary work, easy to add specialists | Routing mistakes by small models → validate the choice |
| **Parallel fan-out / fan-in** (Scenario B) | Independent sub-tasks over different data | Lower latency, clear separation | Merge quality; shared-state key collisions (use separate keys or reducers) |

### Ideas to extend
- Add `MemorySaver` (`compile(checkpointer=...)`) and `interrupt_before=["writer"]` for a human approval step.
- Give Scenario A's specialists their own ReAct tool loops (see the first notebook) instead of fixed fetches.
- Use different models per agent: a tiny one for routing, a bigger one for the writer/editor.
- Nest graphs: use the Situation Room as a single node inside a larger supervisor graph.